In [4]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib
import os

In [5]:
os.makedirs("fastf1_cache", exist_ok=True)

In [6]:
fastf1.Cache.enable_cache("fastf1_cache")

In [7]:
session = fastf1.get_session(2025, 3, 'R')
session.load()

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '44', '6', '23', '87', '14', '22', '10', '55', '7', '27', '30', '31', '5', '18']


In [8]:
laps = session.laps

In [14]:
ver_laps = laps.pick_drivers("VER")

In [15]:
ver_laps = ver_laps[ver_laps["LapTime"].notna()]

In [16]:
## CHECKING!
print(ver_laps[[
    "LapNumber",
    "LapTime",
    "Stint",
    "Compound",
    "PitInTime",
    "PitOutTime"
]].head())

   LapNumber                LapTime  Stint Compound PitInTime PitOutTime
0        1.0 0 days 00:01:34.725000    1.0   MEDIUM       NaT        NaT
1        2.0 0 days 00:01:33.943000    1.0   MEDIUM       NaT        NaT
2        3.0 0 days 00:01:33.639000    1.0   MEDIUM       NaT        NaT
3        4.0 0 days 00:01:33.744000    1.0   MEDIUM       NaT        NaT
4        5.0 0 days 00:01:33.776000    1.0   MEDIUM       NaT        NaT


In [17]:
print(f"Total number of laps for Max Verstappen: {len(ver_laps)}")

Total number of laps for Max Verstappen: 53


## Cleaning laps, i.e., removing the laps that have no effect on the tyre degradation, such as Virtual safety car laps, Flagged laps (yellow/red). Also getting rid of pit in and pit out laps

In [18]:
clean_laps = ver_laps.copy()

In [20]:
## dropping pit-in and pit-out laps
clean_laps = clean_laps[
    clean_laps["PitInTime"].isna() & clean_laps["PitOutTime"].isna()
]

In [21]:
## keeping only green flagged laps (track-status = 1)
clean_laps = clean_laps[clean_laps["TrackStatus"] == '1']

In [29]:
print(clean_laps[
    [
        "LapNumber",
        "LapTime",
        "Stint",
        "Compound",
        "PitInTime",
        "PitOutTime"
    ]
].head(10))

   LapNumber                LapTime  Stint Compound PitInTime PitOutTime
0        1.0 0 days 00:01:34.725000    1.0   MEDIUM       NaT        NaT
1        2.0 0 days 00:01:33.943000    1.0   MEDIUM       NaT        NaT
2        3.0 0 days 00:01:33.639000    1.0   MEDIUM       NaT        NaT
3        4.0 0 days 00:01:33.744000    1.0   MEDIUM       NaT        NaT
4        5.0 0 days 00:01:33.776000    1.0   MEDIUM       NaT        NaT
5        6.0 0 days 00:01:33.646000    1.0   MEDIUM       NaT        NaT
6        7.0 0 days 00:01:33.526000    1.0   MEDIUM       NaT        NaT
7        8.0 0 days 00:01:33.536000    1.0   MEDIUM       NaT        NaT
8        9.0 0 days 00:01:33.529000    1.0   MEDIUM       NaT        NaT
9       10.0 0 days 00:01:33.555000    1.0   MEDIUM       NaT        NaT


In [ ]:
print(f"Raw laps: {ver_laps}")
print(f"Clean laps: {clean_laps}")

## 51 laps in clean lap => 2 laps got cleaned

Raw laps:                      Time Driver DriverNumber                LapTime  \
0  0 days 00:57:41.632000    VER            1 0 days 00:01:34.725000   
1  0 days 00:59:15.575000    VER            1 0 days 00:01:33.943000   
2  0 days 01:00:49.214000    VER            1 0 days 00:01:33.639000   
3  0 days 01:02:22.958000    VER            1 0 days 00:01:33.744000   
4  0 days 01:03:56.734000    VER            1 0 days 00:01:33.776000   
5  0 days 01:05:30.380000    VER            1 0 days 00:01:33.646000   
6  0 days 01:07:03.906000    VER            1 0 days 00:01:33.526000   
7  0 days 01:08:37.442000    VER            1 0 days 00:01:33.536000   
8  0 days 01:10:10.971000    VER            1 0 days 00:01:33.529000   
9  0 days 01:11:44.526000    VER            1 0 days 00:01:33.555000   
10 0 days 01:13:18.024000    VER            1 0 days 00:01:33.498000   
11 0 days 01:14:51.667000    VER            1 0 days 00:01:33.643000   
12 0 days 01:16:25.307000    VER            1 0 days 0